In [1]:
#test

In [2]:
import datetime
import logging  # ログ制御用
import warnings  # 警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Node.js Theme Palette)
# =====================================================================
def show_market_performances(market_dict, title="Market Performance Overview"):
    today = datetime.date.today()
    # 1年分（約370日）のデータを取得して過去騰落率を正確に計算
    start_date = today - datetime.timedelta(days=370)
    
    # Node.js テーマ風のタイトルヘッダー (Dark Background & Neon Green Text)
    title_html = f"""
    <div style='
        background-color: #181818; 
        border-left: 5px solid #68a063; 
        padding: 12px 18px; 
        border-radius: 4px; 
        margin-bottom: 25px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    '>
        <h2 style='
            margin: 0; 
            color: #539e43; 
            font-family: "Consolas", "Courier New", Monaco, monospace; 
            font-size: 22px;
            letter-spacing: 0.5px;
        '>⬢ {title}</h2>
        <span style='
            font-size: 12px; 
            color: #888888; 
            font-family: monospace;
        '>System Status: ONLINE | As of {today.strftime('%Y-%m-%d')}</span>
    </div>
    """
    display(HTML(title_html))
    
    # 数値フォーマット設定
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # Node.jsテーマに合わせたハイライト関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        # プラス（上昇）：Node.jsのネオングリーン背景
        if val >= 10:
            return 'background-color: #1e3a1e; color: #41d888; font-weight: bold; border-radius: 2px;'
        # マイナス（下落）：ダークレッド/コーラル系
        elif val <= -10:
            return 'background-color: #3b1818; color: #ff6b6b; font-weight: bold; border-radius: 2px;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # セクションヘッダー（Node Dark Gray + Green Accent）
        section_html = f"""
        <div style='
            margin-top: 20px; 
            margin-bottom: 8px; 
            color: #215732; 
            font-family: "Consolas", monospace; 
            font-weight: bold;
            font-size: 15px;
            border-bottom: 1px solid #333333;
            padding-bottom: 4px;
        '>
            <span style='color: #68a063;'>❯</span> {market_name}
        </div>
        """
        display(HTML(section_html))
        
        for name, ticker in ticker_dict.items():
            try:
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # テーブル全体のスタイル定義（Dark Node UI Style）
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .set_table_styles([
                             # テーブル全体
                             {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', 'Consolas, monospace'), ('font-size', '13px'), ('background-color', '#121212'), ('color', '#e0e0e0')]},
                             # ヘッダー行（Dark Gray / Green Accent）
                             {'selector': 'th', 'props': [('background-color', '#1f1f1f'), ('color', '#68a063'), ('border', '1px solid #333'), ('padding', '8px 12px'), ('text-align', 'center')]},
                             # データセル
                             {'selector': 'td', 'props': [('border', '1px solid #282828'), ('padding', '7px 12px'), ('text-align', 'right')]},
                             # ホバー効果
                             {'selector': 'tr:hover', 'props': [('background-color', '#1a261a')]}
                         ])
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [3]:
MARKET_TICKERS = {
    # ==========================================
    # 1. アメリカ市場（米国株・指数・ETF）
    # ==========================================
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG",
        "Vanguard FTSE Global All-Cap UCITS ETF (VALL.L)": "VALL.L",
    },

    # ==========================================
    # 2. 日本市場（日本株・指数）
    # ==========================================
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Japan Stock (Custom)": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },

    # ==========================================
    # 3. シンガポール市場（総合・インデックス）
    # ==========================================
    "Singapore (General)": {
        "STI Index": "^STI",
        "DBS Group (金融要塞/東南アジア最大手)": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
        "SGX Group (完全独占/国内唯一の証券取引所)": "S68.SI",
        "PropNex (不動産仲介最大手/高配当)": "OYY.SI",
    },

    # ==========================================
    # 4. シンガポール：医療セクター (Healthcare)
    # ==========================================
    "Singapore Healthcare": {
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone Holdings (クリーンルーム用手袋)": "AP4.SI",
    },

    # ==========================================
    # 5. シンガポール：食品・アグリビジネス
    # ==========================================
    "Singapore Food & Agribusiness": {
        "Wilmar International (世界最大級のパーム油)": "F34.SI",
        "Olam Group (ココア・コーヒー・ナッツ等大手)": "VC2.SI",
    },

    # ==========================================
    # 6. シンガポール：コア・安定・高配当・バリュー
    # ==========================================
    "Singapore Core & Stability": {
        "Sembcorp Industries (大手公益/エネルギー)": "U96.SI",
        "Valuetronics Holdings (電子機器製造受託/無借金)": "BN2.SI",
    },
    "Singapore High Yield & Growth": {
        "Multi-Chem (ITセキュリティ/高配当)": "AWZ.SI",
        "Yangzijiang Financial Holding (金融)": "YF8.SI",
        "Yangzijiang Shipbuilding (造船/高収益)": "BS6.SI",
        "Sanli Environmental Limited": "1E3.SI",
    },
    "Singapore Recovery & Value": {
        "Micro-Mechanics (無借金/半導体消耗工具/高配当)": "508.SI",
    },

    # ==========================================
    # 7. シンガポール：建設・インフラ
    # ==========================================
    "Singapore Construction & Infrastructure": {
        "BRC Asia (鉄筋加工最大手)": "BEC.SI",
        "OKP Holdings (道路土木インフラ/無借金)": "5CF.SI",
        "ISOTeam (HDB建物修繕塗装R&R)": "5WF.SI",
    },
}



In [4]:
show_market_performances(MARKET_TICKERS, title="Market Performance Overview")

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,340.53",-1.03%,-3.32%,-2.40%,-4.82%,-1.24%,+10.01%
NASDAQ Composite,^IXIC,"24,328.38",-0.74%,-4.09%,-2.94%,-7.93%,-7.39%,+7.14%
Apple,AAPL,351.91,-0.25%,-0.84%,+1.05%,+3.73%,+16.62%,+32.03%
NVIDIA,NVDA,155.46,-0.44%,-6.46%,-7.95%,-28.18%,-30.27%,-16.49%
Microsoft,MSFT,506.24,-2.27%,-3.20%,-5.05%,-4.82%,+20.72%,+0.57%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,95.26,+0.12%,+0.04%,-0.65%,+0.35%,-0.98%,-0.83%
Vanguard FTSE Global All-Cap UCITS ETF (VALL.L),VALL.L,3.56,-2.29%,-3.69%,-1.77%,-3.47%,-28.82%,-28.82%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"58,860.43",-1.34%,-3.22%,-2.64%,-9.33%,-4.15%,+14.78%
Toyota Motor,7203.T,"3,414.00",-0.41%,+0.18%,+0.00%,+10.06%,+12.55%,+10.19%
Sony Group,6758.T,"4,271.00",-1.13%,-0.23%,+0.80%,+7.85%,+20.01%,-7.73%
Mitsubishi UFJ Financial,8306.T,"3,287.00",-3.75%,-6.03%,-4.31%,-5.49%,+13.69%,+38.52%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"5,958.00",-1.85%,-4.57%,-2.81%,-17.89%,-6.91%,+16.29%
ニデック (6594),6594.T,"1,668.00",-2.28%,-6.71%,-3.92%,-6.76%,-35.17%,-25.80%
未来工業 (7931),7931.T,"3,260.00",-2.54%,-3.98%,-5.64%,-8.43%,+7.64%,+1.05%
東部ネットワーク (9036),9036.T,"1,111.00",-0.63%,+2.11%,+1.18%,-6.24%,-12.83%,+10.50%
ニトリHD (9843),9843.T,"3,687.00",+0.38%,+7.62%,+8.28%,+10.62%,+53.50%,+48.91%
MRK HD (9980),9980.T,86.00,-1.15%,+2.38%,+2.38%,-3.37%,-14.00%,-6.65%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,109.13",-1.43%,-3.29%,-2.84%,-5.60%,+1.27%,+12.45%
DBS Group (金融要塞/東南アジア最大手),D05.SI,67.70,-2.31%,-3.55%,-2.68%,-8.02%,+12.91%,+31.83%
UOB,U11.SI,36.70,-1.82%,-2.83%,-1.82%,-11.86%,-0.27%,+12.16%
Singtel,Z74.SI,4.10,+2.50%,-1.20%,+0.24%,-1.91%,-16.45%,-12.58%
CapitaLand Ascendas REIT,A17U.SI,2.24,-0.44%,+1.36%,-0.44%,+2.75%,-8.30%,-15.29%
SGX Group (完全独占/国内唯一の証券取引所),S68.SI,20.10,-1.90%,-2.58%,-2.01%,-1.57%,-6.43%,+22.30%
PropNex (不動産仲介最大手/高配当),OYY.SI,1.57,-3.68%,-4.85%,-5.42%,-7.65%,-12.26%,-26.11%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.84,+0.60%,-1.18%,+0.00%,+2.44%,-12.04%,-12.06%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.45,-2.00%,+0.82%,+0.41%,-2.39%,-13.63%,-1.06%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,-4.26%,-6.25%,-6.25%,-19.64%,-23.73%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,3.99,-0.50%,-0.99%,-0.99%,-0.99%,+1.60%,+4.06%
First REIT (ファースト・リート),AW9U.SI,0.16,-4.65%,-6.29%,-6.29%,-7.49%,-26.79%,-34.88%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.46,+0.00%,-5.21%,-6.19%,-14.15%,-23.62%,-8.51%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.18,+0.00%,+0.00%,+0.00%,+0.31%,+20.51%,+54.65%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,+7.50%,+7.50%,+7.50%,+1.52%,-3.71%
Medtecs International (医療用消耗品),546.SI,0.09,-5.00%,+0.00%,-7.77%,-19.49%,-28.03%,-22.76%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,-4.11%,+9.38%,+4.41%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (世界最大級のパーム油),F34.SI,2.94,+1.38%,-0.34%,-0.34%,-14.29%,-20.10%,-8.41%
Olam Group (ココア・コーヒー・ナッツ等大手),VC2.SI,0.84,-0.59%,-1.75%,-2.89%,-7.69%,-25.12%,-5.70%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Sembcorp Industries (大手公益/エネルギー),U96.SI,5.63,-0.88%,-2.26%,-1.92%,-4.74%,-7.21%,-9.34%
Valuetronics Holdings (電子機器製造受託/無借金),BN2.SI,0.92,+0.00%,-3.68%,+0.00%,-5.67%,-4.07%,+10.10%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Multi-Chem (ITセキュリティ/高配当),AWZ.SI,2.45,+0.82%,+1.66%,+1.66%,+7.46%,-30.28%,-20.36%
Yangzijiang Financial Holding (金融),YF8.SI,0.20,+0.00%,-2.38%,+0.00%,+7.33%,-16.33%,-60.19%
Yangzijiang Shipbuilding (造船/高収益),BS6.SI,4.32,-2.48%,-1.82%,-5.88%,-16.12%,+13.39%,+32.45%
Sanli Environmental Limited,1E3.SI,0.07,-4.29%,-12.99%,-14.10%,-28.72%,-62.60%,-77.68%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Micro-Mechanics (無借金/半導体消耗工具/高配当),508.SI,0.50,+0.00%,+0.00%,+0.00%,+0.00%,-35.06%,-14.50%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
BRC Asia (鉄筋加工最大手),BEC.SI,3.91,-2.98%,-5.10%,-6.01%,-2.31%,-15.02%,-1.91%
OKP Holdings (道路土木インフラ/無借金),5CF.SI,0.57,-1.72%,-8.06%,-7.32%,-8.80%,-31.96%,-0.76%
ISOTeam (HDB建物修繕塗装R&R),5WF.SI,0.05,+0.00%,-5.45%,-5.45%,+0.19%,-30.13%,-32.78%


# Gemini Prompt
以下の【分析データ】を基に、指定の【Pythonコードフォーマット】に変換して出力してください。

【出力条件】
・MARKET_TICKERS の辞書形式で出力すること。
・辞書のキーには「企業名 (ティッカー・主要特徴/指標の要約)」を含めること。
・値（Value）には「ティッカーシンボル（.SI付き）」を入れること。
・解説文や導入の挨拶は不要で、そのままコピーして使えるPythonコードのみを出力すること。

【Pythonコードフォーマット例】
MARKET_TICKERS = {
    "カテゴリ名": {
        "銘柄名 (ティッカー・特徴要約)": "ティッカー.SI",
    },
}

【分析データ】
（※ここに分析したテキストや銘柄リスト、あるいは「以下の銘柄を分析してコード化して」と入力）